In [2]:
# Imports
import pandas as pd
from pybaseball import statcast
import numpy as np

# Hide warnings
import warnings
warnings.filterwarnings("ignore")

# Data Download

In [3]:
df_20 = statcast(start_dt="2020-07-23", end_dt="2020-10-27")

This is a large query, it may take a moment to complete


100%|██████████| 97/97 [00:52<00:00,  1.86it/s]


In [4]:
df_21 = statcast(start_dt="2021-04-21", end_dt="2021-10-03")

This is a large query, it may take a moment to complete


100%|██████████| 166/166 [01:40<00:00,  1.65it/s]


In [5]:
df_22 = statcast(start_dt="2022-04-07", end_dt="2022-10-05")

This is a large query, it may take a moment to complete


100%|██████████| 182/182 [01:53<00:00,  1.60it/s]


In [6]:
df_23 = statcast(start_dt="2023-03-30", end_dt="2023-11-01")

This is a large query, it may take a moment to complete


100%|██████████| 217/217 [02:09<00:00,  1.68it/s]


In [7]:
df_24 = statcast(start_dt="2024-03-20", end_dt="2024-10-30")

This is a large query, it may take a moment to complete


100%|██████████| 225/225 [00:19<00:00, 11.75it/s]


In [8]:
df_25 = statcast(start_dt="2025-03-18", end_dt="2025-11-01")

This is a large query, it may take a moment to complete


100%|██████████| 229/229 [00:56<00:00,  4.06it/s]


In [9]:
df_26 = statcast(start_dt='2026-03-25', end_dt="2026-10-28")

This is a large query, it may take a moment to complete


100%|██████████| 218/218 [00:16<00:00, 13.27it/s]


In [10]:
# Print length of all dfs
print("2020: ", len(df_20))
print("2021: ", len(df_21))
print("2022: ", len(df_22))
print("2023: ", len(df_23))
print("2024: ", len(df_24))
print("2025: ", len(df_25))
print("2026: ", len(df_26))

2020:  280398
2021:  638465
2022:  710210
2023:  732562
2024:  745350
2025:  756325
2026:  728343


In [11]:
train = pd.concat([df_20, df_21, df_22, df_23, df_24, df_25, df_26], ignore_index=True)

In [12]:
len(train)

4591653

# Filter Columns & Handle NAs

In [72]:
# Define columns of interest categorically

# Miscellaneous columns not used in modelling
for_test = ['game_year', 'game_date', 'batter', 'player_name', 'pitcher', 'pitch_type', 'inning', 'inning_topbot', 'home_team', 'away_team']
 
# Game situation
situation = ['balls', 'strikes', 'outs_when_up', 'on_1b', 'on_2b', 'on_3b', 'p_throws', 'stand']

# Pitch location
location = ['plate_x', 'plate_z', 'sz_top', 'sz_bot']

# Pitch metrics
pitch = ['release_speed', 'release_pos_x', 'release_pos_z', 'pfx_x', 'pfx_z', 'release_spin_rate', 'release_extension']

# What happened
result = ['description']

# Change in run expectancy
target = ['delta_run_exp']

# Combine
cols = for_test + situation + location + pitch + result + target

df = train[cols]

In [73]:
# Find batter names
from pybaseball import playerid_reverse_lookup

ids = df["batter"].unique().tolist()
names = playerid_reverse_lookup(ids, key_type="mlbam")

names["batter_name"] = names["name_first"].str.title() + " " + names["name_last"].str.title()

df = df.merge(
    names[["key_mlbam", "batter_name"]],
    left_on="batter", right_on="key_mlbam", how="left"
).drop(columns="key_mlbam")

In [74]:
# Find batting and pitching team
top = df["inning_topbot"] == "Top"

df["bat_team"] = np.where(top, df["away_team"], df["home_team"])
df["pitch_team"] = np.where(top, df["home_team"], df["away_team"])

In [75]:
# Display all columns
pd.set_option('display.max_columns', None)

df.tail()

,game_year,game_date,batter,player_name,pitcher,pitch_type,inning,inning_topbot,home_team,away_team,balls,strikes,outs_when_up,on_1b,on_2b,on_3b,p_throws,stand,plate_x,plate_z,sz_top,sz_bot,release_speed,release_pos_x,release_pos_z,pfx_x,pfx_z,release_spin_rate,release_extension,description,delta_run_exp,batter_name,bat_team,pitch_team
4591648,2026,2026-03-25,663757,"Webb, Logan",657277,CH,1,Top,SF,NYY,1,2,0,<NA>,<NA>,<NA>,R,L,0.62265,1.233533,3.138386,1.583858,86.9,-1.97,4.78,-0.99,-0.43,1491,7.0,foul,0.0,Trent Grisham,NYY,SF
4591649,2026,2026-03-25,663757,"Webb, Logan",657277,CH,1,Top,SF,NYY,1,2,0,<NA>,<NA>,<NA>,R,L,-0.09506,1.51478,3.138386,1.583858,87.1,-2.04,4.81,-1.11,-0.55,1582,6.8,foul,0.0,Trent Grisham,NYY,SF
4591650,2026,2026-03-25,663757,"Webb, Logan",657277,FF,1,Top,SF,NYY,0,2,0,<NA>,<NA>,<NA>,R,L,-0.53002,3.911511,3.138386,1.583858,92.9,-1.72,5.04,-1.04,0.8,2062,6.9,ball,0.024,Trent Grisham,NYY,SF
4591651,2026,2026-03-25,663757,"Webb, Logan",657277,CH,1,Top,SF,NYY,0,1,0,<NA>,<NA>,<NA>,R,L,0.304947,1.310579,3.138386,1.583858,85.4,-1.99,4.83,-0.95,-0.66,1577,6.8,swinging_strike,-0.06,Trent Grisham,NYY,SF
4591652,2026,2026-03-25,663757,"Webb, Logan",657277,SI,1,Top,SF,NYY,0,0,0,<NA>,<NA>,<NA>,R,L,-0.109765,3.184813,3.138386,1.583858,92.7,-1.84,5.07,-1.9,0.25,1926,6.8,called_strike,-0.036,Trent Grisham,NYY,SF


In [76]:
df.isna().sum()

game_year                  0
game_date                  0
batter                     0
player_name                0
pitcher                    0
pitch_type             34240
inning                     0
inning_topbot              0
home_team                  0
away_team                  0
balls                      0
strikes                    0
outs_when_up               0
on_1b                3178249
on_2b                3718740
on_3b                4154020
p_throws                   0
stand                      0
plate_x                34450
plate_z                34450
sz_top                 34450
sz_bot                 34450
release_speed          33927
release_pos_x          34146
release_pos_z          34146
pfx_x                  34084
pfx_z                  33940
release_spin_rate      49618
release_extension      39029
description                0
delta_run_exp          14925
batter_name             7468
bat_team                   0
pitch_team                 0
dtype: int64

In [77]:
# Columns that can't be NA
cols_not_na = ['plate_x', 'plate_z', 'sz_top', 'sz_bot', 'delta_run_exp']

df1 = df.dropna(subset=cols_not_na).copy()

In [78]:
len(df1)

4557203

In [79]:
df1.isna().sum()

game_year                  0
game_date                  0
batter                     0
player_name                0
pitcher                    0
pitch_type                 0
inning                     0
inning_topbot              0
home_team                  0
away_team                  0
balls                      0
strikes                    0
outs_when_up               0
on_1b                3151519
on_2b                3698278
on_3b                4128324
p_throws                   0
stand                      0
plate_x                    0
plate_z                    0
sz_top                     0
sz_bot                     0
release_speed             47
release_pos_x             45
release_pos_z             45
pfx_x                    193
pfx_z                     49
release_spin_rate      15453
release_extension       4866
description                0
delta_run_exp              0
batter_name             4763
bat_team                   0
pitch_team                 0
dtype: int64

# Feature Engineering

### Runners On Base

In [80]:
# 1 if runner is on base, 0 if not
base_cols = ["on_1b", "on_2b", "on_3b"]
df1[base_cols] = df1[base_cols].notna().astype(int)

### Pitcher / Batter Sides

In [81]:
# 1 if right-handed, 0 if left-handed
hand_cols = ["stand", "p_throws"]
df1[hand_cols] = (df1[hand_cols] == "R").astype(int)

### Location Relative to SZ Boundary

In [82]:
# `z_diff`: The difference between `plate_z` and either `sz_top` or `sz_bottom` - whichever it is closest to
# Positive value: Pitch landed OUTSIDE the nearest strike zone vertical boundary (Above top edge or below bottom edge)
# Negative value: Pitch landed INSIDE the nearest strike zone vertical boundary (Above the bottom edge or below the top edge)
df1["z_diff"] = np.maximum(df1["plate_z"]-df1["sz_top"], df1["sz_bot"]-df1["plate_z"])

### Swing

In [83]:
df1["description"].unique()

array(['called_strike', 'swinging_strike', 'ball', 'foul',
       'hit_into_play', 'blocked_ball', 'swinging_strike_blocked',
       'foul_bunt', 'foul_tip', 'hit_by_pitch', 'missed_bunt', 'pitchout',
       'bunt_foul_tip', 'foul_pitchout', 'swinging_pitchout'],
      dtype=object)

In [84]:
# Outcomes that are produced with a swing
swing = ['swinging_strike', 'foul', 'hit_into_play', 'swinging_strike_blocked', 'foul_bunt', 'foul_tip', 'missed_bunt', 'bunt_foul_tip', 'foul_pitchout', 'swinging_pitchout']

In [85]:
# 1 if swing occurred, 0 if not
df1["swing"] = df1["description"].isin(swing).astype(int)

In [86]:
# Opposite swing decision
df1["opp_swing"] = (df1["swing"] != 1).astype(int)

# Save as Parquet

In [87]:
# Wanted columns in order
final = df1[['game_year', 'game_date', 'bat_team', 'batter_name', 'batter', 'pitch_team', 'player_name', 'pitcher', 'pitch_type', 'inning',
             'balls', 'strikes', 'outs_when_up', 'on_1b', 'on_2b', 'on_3b',
             'p_throws', 'stand', 'plate_x', 'z_diff',
             'release_speed', 'release_pos_x', 'release_pos_z', 'pfx_x', 'pfx_z',
             'release_spin_rate', 'release_extension',
             'swing', 'opp_swing', 'delta_run_exp']]

In [88]:
# Save as parquet
final.to_parquet("data.parquet")

In [89]:
final

,game_year,game_date,bat_team,batter_name,batter,pitch_team,player_name,pitcher,pitch_type,inning,balls,strikes,outs_when_up,on_1b,on_2b,on_3b,p_throws,stand,plate_x,z_diff,release_speed,release_pos_x,release_pos_z,pfx_x,pfx_z,release_spin_rate,release_extension,swing,opp_swing,delta_run_exp
0,2020,2020-10-27,TB,Willy Adames,642715,LAD,"Urías, Julio",628711,FF,9,0,2,2,0,0,0,0,1,-0.528887,-0.596899,96.7,1.58,5.99,0.18,1.63,2615,5.7,0,1,-0.172
1,2020,2020-10-27,TB,Willy Adames,642715,LAD,"Urías, Julio",628711,FF,9,0,1,2,0,0,0,0,1,-0.553643,-0.456173,94.1,2.91,5.45,0.75,1.3,2470,5.9,0,1,-0.072
2,2020,2020-10-27,TB,Willy Adames,642715,LAD,"Urías, Julio",628711,FF,9,0,0,2,0,0,0,0,1,-0.038806,-0.181556,94.9,1.77,6.02,0.23,1.47,2397,5.7,1,0,-0.043
3,2020,2020-10-27,TB,Mike Brosseau,670712,LAD,"Urías, Julio",628711,FF,9,3,2,1,0,0,0,0,1,-0.370544,-0.619828,94.4,1.66,5.93,0.27,1.5,2508,5.9,0,1,-0.341
4,2020,2020-10-27,TB,Mike Brosseau,670712,LAD,"Urías, Julio",628711,SV,9,2,2,1,0,0,0,0,1,-0.141879,0.569968,81.4,1.46,6.06,-1.67,-0.15,3031,5.7,0,1,0.105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4591648,2026,2026-03-25,NYY,Trent Grisham,663757,SF,"Webb, Logan",657277,CH,1,1,2,0,0,0,0,1,0,0.62265,0.350325,86.9,-1.97,4.78,-0.99,-0.43,1491,7.0,1,0,0.0
4591649,2026,2026-03-25,NYY,Trent Grisham,663757,SF,"Webb, Logan",657277,CH,1,1,2,0,0,0,0,1,0,-0.09506,0.069078,87.1,-2.04,4.81,-1.11,-0.55,1582,6.8,1,0,0.0
4591650,2026,2026-03-25,NYY,Trent Grisham,663757,SF,"Webb, Logan",657277,FF,1,0,2,0,0,0,0,1,0,-0.53002,0.773125,92.9,-1.72,5.04,-1.04,0.8,2062,6.9,0,1,0.024
4591651,2026,2026-03-25,NYY,Trent Grisham,663757,SF,"Webb, Logan",657277,CH,1,0,1,0,0,0,0,1,0,0.304947,0.273279,85.4,-1.99,4.83,-0.95,-0.66,1577,6.8,1,0,-0.06
